# TCC: Previsão de Inadimplência em Concessão de Crédito

**Metodologia:** CRISP-DM

---

## Fases 1 e 2: Entendimento do Negócio e Entendimento dos Dados (*Data Understanding*)

Este *notebook* inicial tem como objetivo a exploração e a compreensão do conjunto de dados oriundo da plataforma *Lending Club*. O foco aqui não é a modelagem, mas sim entender a estrutura das variáveis, a distribuição da nossa classe alvo (inadimplentes vs. adimplentes) e mapear a qualidade dos dados (valores nulos, tipos de dados e anomalias).

---
### 1. Carga Inicial e Inspeção da Estrutura
Nesta etapa, realizamos a importação da base de dados otimizada. Utilizaremos a biblioteca **Polars** para carregar o conjunto de dados previamente estruturado no formato **Parquet**. 

Essa decisão arquitetural permite o processamento de toda a volumetria histórica de dados (eliminando a necessidade de amostragem reduzida) e garante a integridade da tipagem das variáveis financeiras para as etapas de modelagem.

In [ ]:
%pip install polars pyarrow

In [18]:
import polars as pl

# Caminho do arquivo Parquet JÁ PROCESSADO (pós-limpeza e filtros de status)
caminho_parquet = r'DataSet_Loan\base_credito_tcc.parquet'

# Carga instantânea dos dados para a memória
df = pl.read_parquet(caminho_parquet)

# Exibindo as primeiras linhas para validação visual da estrutura
df.head()

loan_amnt,term,int_rate,grade,annual_inc,dti,emp_length,home_ownership,fico_range_low,fico_range_high,open_acc,pub_rec,revol_util,total_acc,mort_acc,inq_last_6mths,pub_rec_bankruptcies,target
f64,str,f64,str,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
3600.0,""" 36 months""",13.99,"""C""",55000.0,5.91,"""10+ years""","""MORTGAGE""",675.0,679.0,7.0,0.0,29.7,13.0,1.0,1.0,0.0,0
24700.0,""" 36 months""",11.99,"""C""",65000.0,16.06,"""10+ years""","""MORTGAGE""",715.0,719.0,22.0,0.0,19.2,38.0,4.0,4.0,0.0,0
20000.0,""" 60 months""",10.78,"""B""",63000.0,10.78,"""10+ years""","""MORTGAGE""",695.0,699.0,6.0,0.0,56.2,18.0,5.0,0.0,0.0,0
10400.0,""" 60 months""",22.45,"""F""",104433.0,25.37,"""3 years""","""MORTGAGE""",695.0,699.0,12.0,0.0,64.5,35.0,6.0,3.0,0.0,0
11950.0,""" 36 months""",13.44,"""C""",34000.0,10.2,"""4 years""","""RENT""",690.0,694.0,5.0,0.0,68.4,6.0,0.0,0.0,0.0,0


### 1.1. Dimensões, Tipagem e Integridade dos Dados
Com a base consolidada em memória, verificamos o volume total de registros retidos após a limpeza inicial, o schema de dados inferido e a presença de dados faltantes (nulos).

In [7]:
# 1. Dimensões da Base
print(f"Total de Clientes (Linhas): {df.height}")
print(f"Total de Atributos (Colunas): {df.width}")

# 2. Valores Nulos
print("\n--- Contagem de Valores Nulos ---")
print(df.null_count())

# 3. Tipagem (Schema)
print("\n--- Tipagem das Colunas ---")
for coluna, tipo in df.schema.items():
    print(f"{coluna}: {tipo}")

Total de Clientes (Linhas): 1345310
Total de Atributos (Colunas): 18

--- Contagem de Valores Nulos ---
shape: (1, 18)
┌───────────┬──────┬──────────┬───────┬───┬──────────┬────────────────┬───────────────────┬────────┐
│ loan_amnt ┆ term ┆ int_rate ┆ grade ┆ … ┆ mort_acc ┆ inq_last_6mths ┆ pub_rec_bankruptc ┆ target │
│ ---       ┆ ---  ┆ ---      ┆ ---   ┆   ┆ ---      ┆ ---            ┆ ies               ┆ ---    │
│ u32       ┆ u32  ┆ u32      ┆ u32   ┆   ┆ u32      ┆ u32            ┆ ---               ┆ u32    │
│           ┆      ┆          ┆       ┆   ┆          ┆                ┆ u32               ┆        │
╞═══════════╪══════╪══════════╪═══════╪═══╪══════════╪════════════════╪═══════════════════╪════════╡
│ 0         ┆ 0    ┆ 0        ┆ 0     ┆ … ┆ 47281    ┆ 1              ┆ 697               ┆ 0      │
└───────────┴──────┴──────────┴───────┴───┴──────────┴────────────────┴───────────────────┴────────┘

--- Tipagem das Colunas ---
loan_amnt: Float64
term: String
int_rate: Fl

### 1.2. Distribuição da Variável Alvo (Target)
O mapeamento da proporção entre bons pagadores (0) e inadimplentes (1) é essencial para validar o desbalanceamento natural do cenário de concessão de crédito, fator que guiará as estratégias de validação (como a estratificação) e a escolha de métricas de avaliação (Recall, F1-Score).

In [17]:
print("Distribuição de Inadimplência:\n")

distribuicao = (
    df['target'].value_counts(normalize=True)
    .with_columns((pl.col('proportion') * 100).alias('porcentagem (%)'))
)

print(distribuicao)

Distribuição de Inadimplência:

shape: (2, 3)
┌────────┬────────────┬─────────────────┐
│ target ┆ proportion ┆ porcentagem (%) │
│ ---    ┆ ---        ┆ ---             │
│ i8     ┆ f64        ┆ f64             │
╞════════╪════════════╪═════════════════╡
│ 0      ┆ 0.800374   ┆ 80.037389       │
│ 1      ┆ 0.199626   ┆ 19.962611       │
└────────┴────────────┴─────────────────┘
